In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from utilsforecast.plotting import plot_series
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import *

import warnings
warnings.filterwarnings("ignore")

In [2]:
# df = pd.read_csv("https://raw.githubusercontent.com/marcopeix/youtube_tutorials/refs/heads/main/data/daily_sales_french_bakery.csv", parse_dates = ["ds"])

df = pd.read_csv("daily_sales_french_bakery.csv", parse_dates=["ds"])
df.head()

,unique_id,ds,y,unit_price
0,12 MACARON,2022-07-13,10.0,10.0
1,12 MACARON,2022-07-14,0.0,10.0
2,12 MACARON,2022-07-15,0.0,10.0
3,12 MACARON,2022-07-16,10.0,10.0
4,12 MACARON,2022-07-17,30.0,10.0


In [3]:
df = df.groupby("unique_id").filter(lambda x: len(x) >= 28)
df.head()

,unique_id,ds,y,unit_price
0,12 MACARON,2022-07-13,10.0,10.0
1,12 MACARON,2022-07-14,0.0,10.0
2,12 MACARON,2022-07-15,0.0,10.0
3,12 MACARON,2022-07-16,10.0,10.0
4,12 MACARON,2022-07-17,30.0,10.0


In [4]:
df = df.drop(["unit_price"], axis=1
             )

In [5]:
df.head()

,unique_id,ds,y
0,12 MACARON,2022-07-13,10.0
1,12 MACARON,2022-07-14,0.0
2,12 MACARON,2022-07-15,0.0
3,12 MACARON,2022-07-16,10.0
4,12 MACARON,2022-07-17,30.0


In [6]:
df["unique_id"].value_counts()

unique_id
BAGUETTE              637
BOULE 400G            637
BANETTE               637
BOULE 200G            637
BANETTINE             637
                     ... 
RELIGIEUSE             71
SABLE F  P             68
DELICETROPICAL         63
VIENNOISE              61
PAIN SUISSE PEPITO     51
Name: count, Length: 121, dtype: int64

In [7]:
plot_series(df=df, ids=["BAGUETTE", "CROISSANT"], palette="tab10", engine="plotly")

In [8]:
plot_series(df=df, ids=["BAGUETTE", "CROISSANT"], max_insample_length=56, palette="tab10", engine="plotly")

In [9]:
# Baseline models
from statsforecast import StatsForecast
from statsforecast.models import Naive, HistoricAverage, WindowAverage, SeasonalNaive

In [10]:
horizon = 7

models = [
    Naive(),
    HistoricAverage(),
    WindowAverage(window_size=7),
    SeasonalNaive(season_length=7)
]

sf = StatsForecast(models=models, freq="D")
sf.fit(df=df)
preds = sf.predict(h=horizon)

In [11]:
preds.head()

,unique_id,ds,Naive,HistoricAverage,WindowAverage,SeasonalNaive
0,12 MACARON,2022-09-29,10.0,8.974359,2.857143,0.0
1,12 MACARON,2022-09-30,10.0,8.974359,2.857143,0.0
2,12 MACARON,2022-10-01,10.0,8.974359,2.857143,10.0
3,12 MACARON,2022-10-02,10.0,8.974359,2.857143,0.0
4,12 MACARON,2022-10-03,10.0,8.974359,2.857143,0.0


In [12]:
plot_series(
    df=df,
    forecasts_df=preds,
    ids=["BAGUETTE", "CROISSANT"],
    max_insample_length=28,
    palette="tab10",
    engine="plotly"
)

### Evaluate baseline models

In [13]:
test = df.groupby("unique_id").tail(7)
train = df.drop(test.index).reset_index(drop=True)

In [14]:
sf.fit(df = train)
preds = sf.predict(h=horizon)
eval_df = pd.merge(test, preds, "left", ['ds', 'unique_id'])

In [15]:
evaluation = evaluate(
    eval_df,
    metrics=[mae], 
)

evaluation.head()

,unique_id,metric,Naive,HistoricAverage,WindowAverage,SeasonalNaive
0,12 MACARON,mae,2.857143,6.961771,3.469388,4.285714
1,BAGUETTE,mae,17.142857,5.455193,7.877551,12.571429
2,BAGUETTE APERO,mae,0.000000,0.537572,0.642857,0.642857
3,BAGUETTE GRAINE,mae,9.800000,4.612271,2.942857,0.200000
4,BANETTE,mae,1.314286,5.421984,6.008163,7.885714


In [16]:
evaluation = evaluation.drop(['unique_id'],  axis=1).groupby('metric').mean().reset_index()
evaluation

,metric,Naive,HistoricAverage,WindowAverage,SeasonalNaive
0,mae,6.107556,5.228439,5.011663,4.613636


In [17]:
import plotly.express as px
import pandas as pd

# 1. Extract your data (same as your original code)
methods = evaluation.columns[1:].tolist()
values = evaluation.iloc[0, 1:].tolist()

# 2. Plotly loves DataFrames, so we quickly package the lists into one
df_plot = pd.DataFrame({
    'Method': methods,
    'Mean absolute error (MAE)': values
})

# 3. Create the interactive bar chart
fig = px.bar(
    df_plot, 
    x='Method', 
    y='Mean absolute error (MAE)',
    text='Mean absolute error (MAE)' # This replaces your plt.text() loop!
)

# 4. Format the text to 3 decimal places and position it above the bars
fig.update_traces(
    texttemplate='%{text:.3f}', 
    textposition='outside'
)

# 5. Make the layout look clean
fig.update_layout(
    title="Model Evaluation (MAE Comparison)",
    yaxis_title="Mean absolute error (MAE)"
)

fig.show()

### AutoARIMA

In [18]:
from statsforecast.models import AutoARIMA

In [19]:
unique_ids = ["BAGUETTE", "CROISSANT"]

small_train = train[train['unique_id'].isin(unique_ids)]
small_test = test[test['unique_id'].isin(unique_ids)]

models = [
    AutoARIMA(seasonal=False, alias="ARIMA"),
    AutoARIMA(season_length=7, alias="SARIMA")
]

sf = StatsForecast(models=models, freq="D")
sf.fit(df=small_train)
arima_preds = sf.predict(h=horizon)

arima_eval_df = pd.merge(arima_preds, eval_df, "inner", ['ds', 'unique_id'])
arima_eval = evaluate(
    arima_eval_df,
    metrics = [mae],
)

arima_eval

,unique_id,metric,ARIMA,SARIMA,Naive,HistoricAverage,WindowAverage,SeasonalNaive
0,BAGUETTE,mae,9.353153,7.449083,17.142857,5.455193,7.877551,12.571429
1,CROISSANT,mae,14.565395,10.359143,17.485714,22.618934,18.244898,12.857143


In [20]:
arima_eval = arima_eval.drop(['unique_id'], axis=1).groupby('metric').mean().reset_index()
arima_eval

,metric,ARIMA,SARIMA,Naive,HistoricAverage,WindowAverage,SeasonalNaive
0,mae,11.959274,8.904113,17.314286,14.037063,13.061224,12.714286


In [21]:
plot_series(
    df=df,
    forecasts_df=arima_preds,
    ids = ["BAGUETTE", "CROISSANT"],
    max_insample_length=28,
    palette="tab10",
    engine="plotly"
)

In [22]:
import plotly.express as px
import pandas as pd

# 1. Extract methods and values (same as your original code)
methods = arima_eval.columns[1:].tolist()
values = arima_eval.iloc[0, 1:].tolist()

# 2. Package into a DataFrame
df_plot = pd.DataFrame({
    'Method': methods,
    'Mean absolute error (MAE)': values
})

# 3. Sort the data (Replacing your manual zip/lambda sorting logic)
# ascending=False matches your reverse=True requirement
df_plot = df_plot.sort_values(by='Mean absolute error (MAE)', ascending=False)

# 4. Create the interactive bar chart
fig = px.bar(
    df_plot, 
    x='Method', 
    y='Mean absolute error (MAE)',
    text='Mean absolute error (MAE)' # Auto-handles the text labels
)

# 5. Format to 3 decimal places and clean up layout
fig.update_traces(
    texttemplate='%{text:.3f}', 
    textposition='outside'
)
fig.update_layout(
    title="ARIMA Models Evaluation (MAE Comparison)",
    yaxis_title="Mean absolute error (MAE)"
)

# 6. Render the interactive chart
fig.show()

### Cross-validation

In [23]:
small_df = df[df["unique_id"].isin(unique_ids)]

models = [
    SeasonalNaive(season_length=7),
    AutoARIMA(seasonal=False, alias="ARIMA"),
    AutoARIMA(season_length=7, alias="SARIMA")
]

sf = StatsForecast(models=models, freq="D")
cv_df = sf.cross_validation(
    h=horizon,   # 7 days
    df=small_df,
    n_windows=8,
    step_size=horizon,
    refit=True
)

cv_df.head()

,unique_id,ds,cutoff,y,SeasonalNaive,ARIMA,SARIMA
0,BAGUETTE,2022-08-06,2022-08-05,55.0,68.0,71.355197,71.584714
1,BAGUETTE,2022-08-07,2022-08-05,67.0,70.0,70.337984,78.458884
2,BAGUETTE,2022-08-08,2022-08-05,61.0,48.0,61.195007,57.001733
3,BAGUETTE,2022-08-09,2022-08-05,52.0,49.0,52.649015,49.401145
4,BAGUETTE,2022-08-10,2022-08-05,57.0,57.0,47.785877,49.725279


In [24]:
plot_series(
    df=small_df, 
    forecasts_df=cv_df.drop(["y", "cutoff"], axis=1), 
    ids=["BAGUETTE", "CROISSANT"],
    max_insample_length=140, 
    palette="tab10",
    engine="plotly"
    )

In [25]:
cv_eval = evaluate(
    cv_df.drop(["cutoff"], axis=1),
    metrics=[mae],
)
cv_eval = cv_eval.drop(['unique_id'], axis=1).groupby('metric').mean().reset_index()
cv_eval

,metric,SeasonalNaive,ARIMA,SARIMA
0,mae,21.117857,21.17093,19.281295


In [26]:
import plotly.express as px
import pandas as pd

# 1. Extract methods and values from the CV dataframe
methods = cv_eval.columns[1:].tolist()
values = cv_eval.iloc[0, 1:].tolist()

# 2. Package into a DataFrame
df_plot = pd.DataFrame({
    'Method': methods,
    'Mean absolute error (MAE)': values
})

# 3. Sort the data (descending order to match reverse=True)
df_plot = df_plot.sort_values(by='Mean absolute error (MAE)', ascending=False)

# 4. Create the interactive bar chart
fig = px.bar(
    df_plot, 
    x='Method', 
    y='Mean absolute error (MAE)',
    text='Mean absolute error (MAE)'
)

# 5. Format to 3 decimal places and add clear titles
fig.update_traces(
    texttemplate='%{text:.3f}', 
    textposition='outside'
)
fig.update_layout(
    title="Cross-Validation Evaluation (MAE Comparison)",
    yaxis_title="Mean absolute error (MAE)"
)

# 6. Render the interactive chart
fig.show()

### Forecasting with exogenous features

In [27]:
# df = pd.read_csv("https://raw.githubusercontent.com/marcopeix/youtube_tutorials/refs/heads/main/data/daily_sales_french_bakery.csv", parse_dates=["ds"])

df = pd.read_csv("daily_sales_french_bakery.csv", parse_dates=["ds"])
df = df.groupby('unique_id').filter(lambda x: len(x) >= 28)
df.head()

,unique_id,ds,y,unit_price
0,12 MACARON,2022-07-13,10.0,10.0
1,12 MACARON,2022-07-14,0.0,10.0
2,12 MACARON,2022-07-15,0.0,10.0
3,12 MACARON,2022-07-16,10.0,10.0
4,12 MACARON,2022-07-17,30.0,10.0


In [28]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. Filter the data (Same as your code)
baguette_plot_df = df[df["unique_id"] == "BAGUETTE"]
croissant_plot_df = df[df["unique_id"] == "CROISSANT"]

# 2. Create the 2x2 subplot grid
# 'shared_xaxes=True' means zooming in on one chart zooms all of them!
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Baguette Sales Volume", "Baguette Unit Price",
                    "Croissant Sales Volume", "Croissant Unit Price"),
    shared_xaxes=True 
)

# 3. Add traces (the actual lines) to specific rows and columns
# ax1 -> row 1, col 1
fig.add_trace(go.Scatter(x=baguette_plot_df['ds'], y=baguette_plot_df['y'], 
                         name="Baguette Volume"), row=1, col=1)

# ax2 -> row 1, col 2
fig.add_trace(go.Scatter(x=baguette_plot_df['ds'], y=baguette_plot_df['unit_price'], 
                         name="Baguette Price"), row=1, col=2)

# ax3 -> row 2, col 1
fig.add_trace(go.Scatter(x=croissant_plot_df['ds'], y=croissant_plot_df['y'], 
                         name="Croissant Volume"), row=2, col=1)

# ax4 -> row 2, col 2
fig.add_trace(go.Scatter(x=croissant_plot_df['ds'], y=croissant_plot_df['unit_price'], 
                         name="Croissant Price"), row=2, col=2)

# 4. Clean up the layout (equivalent to figsize and tight_layout)
fig.update_layout(
    height=500, 
    width=900, 
    title_text="Bakery Sales & Pricing Analysis",
    showlegend=False # Hiding the legend since titles explain the data
)

# Render the interactive dashboard
fig.show()

In [29]:
unique_ids = ["BAGUETTE", "CROISSANT"]
small_df = df[df["unique_id"].isin(unique_ids)]
test = small_df.groupby("unique_id").tail(7)
train = small_df.drop(test.index).reset_index(drop=True)

In [30]:
futr_exog_df = test.drop(["y"], axis=1)
futr_exog_df.head()

,unique_id,ds,unit_price
714,BAGUETTE,2022-09-24,1.0
715,BAGUETTE,2022-09-25,1.0
716,BAGUETTE,2022-09-26,1.0
717,BAGUETTE,2022-09-27,1.0
718,BAGUETTE,2022-09-28,1.0


In [31]:
models = [
    AutoARIMA(season_length=7, alias="SARIMA_exog")
]

sf = StatsForecast(models=models, freq="D")
sf.fit(df=train)
arima_exog_preds = sf.predict(h=7, X_df=futr_exog_df)

models = [
    AutoARIMA(season_length=7, alias="SARIMA")
]

sf = StatsForecast(models=models, freq="D")
sf.fit(df=train.drop(["unit_price"], axis=1))
arima_preds = sf.predict(h=horizon)

test_df = test.merge(arima_exog_preds, on=["unique_id", "ds"], how="left")\
              .merge(arima_preds, on=["unique_id", "ds"], how="left")
test_df

,unique_id,ds,y,unit_price,SARIMA_exog,SARIMA
0,BAGUETTE,2022-09-24,28.0,1.0,28.898656,28.657114
1,BAGUETTE,2022-09-25,36.0,1.0,42.512939,42.182373
2,BAGUETTE,2022-09-26,18.0,1.0,20.423555,20.013653
3,BAGUETTE,2022-09-27,34.0,1.0,19.065851,18.646490
4,BAGUETTE,2022-09-28,23.0,1.0,16.468540,16.114925
5,BAGUETTE,2022-09-29,30.0,1.0,21.656385,21.288018
6,BAGUETTE,2022-09-30,35.0,1.0,23.100802,22.660123
7,CROISSANT,2022-09-24,51.6,1.2,64.424667,64.011003
8,CROISSANT,2022-09-25,109.2,1.2,135.123808,134.488202
9,CROISSANT,2022-09-26,31.2,1.2,43.525613,42.735408


In [32]:
plot_series(
    df=train, 
    forecasts_df=test_df, 
    ids=["BAGUETTE", "CROISSANT"], 
    max_insample_length=28,
    models=["SARIMA_exog", "SARIMA"],
    palette="tab10",
    engine="plotly"
)

In [33]:
models = [
    AutoARIMA(season_length=7, alias="SARIMA_exog")
]

sf = StatsForecast(models=models, freq="D")
cv_exog_df = sf.cross_validation(
    h=horizon, # 7 days
    df=small_df,
    n_windows=8,
    step_size=7,
    refit=True
)

cv_exog_df.head()

,unique_id,ds,cutoff,y,SARIMA_exog
0,BAGUETTE,2022-08-06,2022-08-05,55.0,71.511188
1,BAGUETTE,2022-08-07,2022-08-05,67.0,78.457622
2,BAGUETTE,2022-08-08,2022-08-05,61.0,57.062223
3,BAGUETTE,2022-08-09,2022-08-05,52.0,49.525279
4,BAGUETTE,2022-08-10,2022-08-05,57.0,49.485229


In [34]:
cv_exog_eval = evaluate(
    cv_exog_df.drop(["cutoff"], axis=1),
    metrics=[mae],
)
cv_exog_eval = cv_exog_eval.drop(['unique_id'], axis=1).groupby('metric').mean().reset_index()
cv_exog_eval

,metric,SARIMA_exog
0,mae,19.21056


### Creating features from timestamps

In [35]:
from functools import partial
from utilsforecast.feature_engineering import fourier, time_features, pipeline
features = [
    partial(fourier, season_length=7, k=2),
    partial(time_features, features=["day", "week", "month"])
]

small_exog_df, futr_exog_df = pipeline(
    df=small_df,
    features=features,
    freq="D",
    h=horizon
)

In [36]:

small_exog_df.head()

,unique_id,ds,y,unit_price,sin1_7,sin2_7,cos1_7,cos2_7,day,week,month
84,BAGUETTE,2021-01-02,41.4,0.9,0.781832,0.974928,0.623490,-0.222521,2,53,1
85,BAGUETTE,2021-01-03,31.5,0.9,0.974928,-0.433884,-0.222521,-0.900969,3,53,1
86,BAGUETTE,2021-01-04,27.0,0.9,0.433884,-0.781831,-0.900969,0.623490,4,1,1
87,BAGUETTE,2021-01-05,26.1,0.9,-0.433884,0.781832,-0.900969,0.623490,5,1,1
88,BAGUETTE,2021-01-06,0.0,0.9,-0.974928,0.433884,-0.222521,-0.900969,6,1,1


In [37]:
futr_exog_df

,unique_id,ds,sin1_7,sin2_7,cos1_7,cos2_7,day,week,month
0,BAGUETTE,2022-10-01,0.781844,0.974919,0.623474,-0.222559,1,39,10
1,BAGUETTE,2022-10-02,0.974927,-0.433892,-0.222526,-0.900965,2,39,10
2,BAGUETTE,2022-10-03,0.433893,-0.781844,-0.900964,0.623474,3,40,10
3,BAGUETTE,2022-10-04,-0.433861,0.781800,-0.900980,0.623529,4,40,10
4,BAGUETTE,2022-10-05,-0.974933,0.433846,-0.222500,-0.900987,5,40,10
5,BAGUETTE,2022-10-06,-0.781828,-0.974931,0.623495,-0.222509,6,40,10
6,BAGUETTE,2022-10-07,-0.000009,-0.000017,1.000000,1.000000,7,40,10
7,CROISSANT,2022-10-01,0.781844,0.974919,0.623474,-0.222559,1,39,10
8,CROISSANT,2022-10-02,0.974927,-0.433892,-0.222526,-0.900965,2,39,10
9,CROISSANT,2022-10-03,0.433893,-0.781844,-0.900964,0.623474,3,40,10


In [38]:
models = [
    AutoARIMA(season_length=7, alias="SARIMA_time_exog")
]

sf = StatsForecast(models=models, freq="D")
cv_time_exog_df = sf.cross_validation(
    h=horizon, # 7 days
    df=small_exog_df,
    n_windows=8,
    step_size=horizon,
    refit=True
)

cv_time_exog_eval = evaluate(
    cv_time_exog_df.drop(["cutoff"], axis=1),
    metrics=[mae],
)
cv_time_exog_eval = cv_time_exog_eval.drop(['unique_id'], axis=1).groupby('metric').mean().reset_index()
cv_time_exog_eval

,metric,SARIMA_time_exog
0,mae,19.557084


In [39]:
import plotly.express as px
import pandas as pd

# 1. Define your data arrays
methods = ["ARIMA", "Seasonal naive", "SARIMA", "SARIMA_price_exog", "SARIMA_time_exog"] 
values = [21.229, 21.118, 19.281, 19.210, 19.533]

# 2. Package into a DataFrame
df_plot = pd.DataFrame({
    'Method': methods,
    'Mean absolute error (MAE)': values
})

# 3. Sort the data (descending order matches your reverse=True)
df_plot = df_plot.sort_values(by='Mean absolute error (MAE)', ascending=False)

# 4. Create the interactive bar chart
fig = px.bar(
    df_plot, 
    x='Method', 
    y='Mean absolute error (MAE)',
    text='Mean absolute error (MAE)'
)

# 5. Format to 3 decimal places and add clear titles
fig.update_traces(
    texttemplate='%{text:.3f}', 
    textposition='outside'
)
fig.update_layout(
    title="SARIMA & Exogenous Models Evaluation (MAE)",
    yaxis_title="Mean absolute error (MAE)"
)

# 6. Render the interactive chart
fig.show()

### Prediction intervals

In [40]:
unique_ids = ["BAGUETTE", "CROISSANT"]
small_df = df[df["unique_id"].isin(unique_ids)]
test = small_df.groupby("unique_id").tail(7)
train = small_df.drop(test.index).reset_index(drop=True)

In [41]:
train.head()

,unique_id,ds,y,unit_price
0,BAGUETTE,2021-01-02,41.4,0.9
1,BAGUETTE,2021-01-03,31.5,0.9
2,BAGUETTE,2021-01-04,27.0,0.9
3,BAGUETTE,2021-01-05,26.1,0.9
4,BAGUETTE,2021-01-06,0.0,0.9


In [42]:
models = [
    AutoARIMA(season_length=7)
]

sf = StatsForecast(models=models, freq="D")
sf.fit(df=train)
prob_preds = sf.predict(h=horizon, X_df=test.drop(["y"], axis=1), level=[80])
test_df = test.merge(prob_preds, on=["unique_id", "ds"], how="left")

In [43]:
plot_series(
    df=train, 
    forecasts_df=test_df, 
    ids=["BAGUETTE", "CROISSANT"], 
    max_insample_length=28,
    models=["AutoARIMA"],
    level=[80],
    palette="tab10",
    engine="plotly"
)

In [44]:
models = [
    AutoARIMA(season_length=7)
]

sf = StatsForecast(models=models, freq="D")
cv_prob_df = sf.cross_validation(
    h=horizon,
    df=small_df,
    n_windows=8,
    step_size=7,
    refit=True,
    level=[80],
)

In [45]:
plot_series(
    df=small_df, 
    forecasts_df=cv_prob_df.drop(["y", "cutoff"], axis=1), 
    ids=["BAGUETTE", "CROISSANT"], 
    models=["AutoARIMA"],
    max_insample_length=140,
    level=[80],
    palette="tab10",
    engine="plotly"
)

### Evaluation metrics

In [46]:
models = [
    AutoARIMA(season_length=7, alias="SARIMA_exog"),
    SeasonalNaive(season_length=7)
]

sf = StatsForecast(models=models, freq="D")
final_cv_df = sf.cross_validation(
    h=horizon,
    df=small_df,
    n_windows=8,
    step_size=7,
    refit=True,
    level=[80],
)

In [47]:
final_cv_df.head()

,unique_id,ds,cutoff,y,SARIMA_exog,SARIMA_exog-lo-80,SARIMA_exog-hi-80,SeasonalNaive,SeasonalNaive-lo-80,SeasonalNaive-hi-80
0,BAGUETTE,2022-08-06,2022-08-05,55.0,71.511188,58.278868,84.743508,68.0,50.158042,85.841958
1,BAGUETTE,2022-08-07,2022-08-05,67.0,78.457622,64.353669,92.561575,70.0,52.158042,87.841958
2,BAGUETTE,2022-08-08,2022-08-05,61.0,57.062223,42.621647,71.502798,48.0,30.158042,65.841958
3,BAGUETTE,2022-08-09,2022-08-05,52.0,49.525279,34.846687,64.203872,49.0,31.158042,66.841958
4,BAGUETTE,2022-08-10,2022-08-05,57.0,49.485229,34.606338,64.364119,57.0,39.158042,74.841958


In [48]:
temp_test = small_df.groupby("unique_id").tail(7*8)
temp_train = small_df.drop(temp_test.index).reset_index(drop=True)

In [49]:
models = ["SARIMA_exog", "SeasonalNaive"]
metrics = [
    mae,
    mse, 
    rmse, 
    mape, 
    smape,
    partial(mase, seasonality=7),
    scaled_crps
]

final_eval = evaluate(
    final_cv_df.drop(["ds", "cutoff"], axis=1),
    metrics=metrics,
    models=models,
    train_df=temp_train,
    level=[80]
)
final_eval = final_eval.drop(['unique_id'], axis=1).groupby('metric').mean().reset_index()
final_eval

,metric,SARIMA_exog,SeasonalNaive
0,mae,19.210560,21.117857
1,mape,0.328602,0.376819
2,mase,1.181443,1.328592
3,mse,792.671844,970.417143
4,rmse,24.978061,27.875413
5,scaled_crps,0.153625,0.166451
6,smape,0.168234,0.211317


In [64]:
import plotly.express as px
import pandas as pd

# 1. Reshape the data from Wide to Long format
# This turns columns (SARIMA_exog, SeasonalNaive) into rows categorized by a 'Model' column
df_long = final_eval.melt(
    id_vars=['metric'], 
    value_vars=['SARIMA_exog', 'SeasonalNaive'],
    var_name='Model', 
    value_name='Score'
)

# 2. Create the faceted bar chart
fig = px.bar(
    df_long, 
    x='Model', 
    y='Score', 
    color='Model',          # Colors the bars
    facet_col='metric',     # Creates a separate subplot for each metric
    facet_col_wrap=3,       # Automatically wraps into your 3x3 grid!
    text='Score',           # Automatically handles the bar labels
    color_discrete_sequence=['#1f77b4', '#d62728'] # Similar to your blue/red
)

# 3. Format the numbers and clean up titles
fig.update_traces(
    texttemplate='%{text:.3f}', 
    textposition='outside'
)

# Plotly adds "metric=" to subplot titles by default. This strips it out.
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].upper()))

# ==========================================
# CRITICAL FIX: Unlink the y-axes!
# ==========================================
fig.update_yaxes(matches=None)
fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))

# 4. Final layout adjustments
fig.update_layout(
    height=1300, 
    width=1300, 
    title_text="Final Evaluation Comparison (SARIMA_exog vs SeasonalNaive)",
    showlegend=False # Legend isn't needed since the x-axis labels the models
)

# Render the interactive dashboard
fig.show()